[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aditya-siraskar/PINN_FTRE/blob/main/notebooks/08_phase4_stage7_ablation_release.ipynb) [![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/aditya-siraskar/PINN_FTRE/blob/main/notebooks/08_phase4_stage7_ablation_release.ipynb)

# 08 · Stage 7: physics vs data, failure cases, final scorecard

**Phase:** Phase 4: empirical and thesis

**Goal:** Does the physics help (RQ3)? The Stage 4 PINN, trained on no samples, against the same network fitted to N exact samples by maximum likelihood (N = 10³…10⁶) and against Monte Carlo on those samples. Then the cases where the method breaks, and one scorecard over Stages 2–7 for the thesis.

**Outputs:** `figures/stage7_physics_vs_data_d{2,5}/`, `figures/stage7_failure_<case>/`, `figures/stage7_final/`; same under `results/`

> Runtime → Change runtime type → **T4 GPU**. Run the setup cell first. With `USE_DRIVE = True` the repo lives in
> `MyDrive/PINN_FTRE`, so figures and results persist. Core logic lives in `pinn_tailrisk/` and `scripts/`;
> this notebook only runs it and shows the results.

**On Kaggle** (free GPU, 30 h/week): open with the Kaggle badge. Right panel → Settings: **Accelerator = GPU T4 x2**
(not P100: recent PyTorch builds no longer support it), **Internet = on** (needs a phone-verified account).
Then **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 h, so closing the
browser does not stop it. When it finishes, open the version → **Output** → download `stage7_outputs.zip`.

In [ ]:
# @title Setup: repo on Google Drive (Colab) or /kaggle/working (Kaggle), install, GPU check  { display-mode: "form" }
USE_DRIVE = True   # Colab: keep figures/results across sessions (recommended). Ignored on Kaggle.
REPO_URL = "https://github.com/aditya-siraskar/PINN_FTRE"

import os, sys
ON_KAGGLE = os.path.exists("/kaggle/working")   # needs Settings > Internet on, Accelerator > GPU T4 x2
if ON_KAGGLE:
    BASE = "/kaggle/working"   # kept with the notebook's output when run via Save Version
elif USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
else:
    BASE = "/content"
ROOT = f"{BASE}/PINN_FTRE"
if not os.path.exists(ROOT):
    !git clone -q {REPO_URL} "{ROOT}"
    if not os.path.exists(ROOT):
        raise RuntimeError("git clone failed: no internet. Kaggle: Settings > Internet on (phone-verified account), "
                           "then Run > Restart & clear outputs and run again.")
else:
    rc = os.system(f'git -C "{ROOT}" pull -q --ff-only')
    if rc != 0:   # usually: result files on Drive that have since been committed to GitHub
        print("WARNING: git pull failed; the Drive copy is out of date. To sync (keeps untracked files), run:")
        print(f'!git -C "{ROOT}" fetch -q origin && git -C "{ROOT}" reset -q --hard origin/main')
%cd "{ROOT}"
!pip install -q -e .
sys.path.insert(0, ROOT)
import importlib   # forget a stale "not found" from an earlier run of this cell before the clone existed
sys.path_importer_cache.pop(ROOT, None); importlib.invalidate_caches()

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else
      "none (Colab: Runtime > Change runtime type > T4 GPU; Kaggle: Settings > Accelerator > GPU T4 x2)")

In [ ]:
# autoreload picks up library edits without a restart. Colab's IPython build imports the
# removed `imp` module on Python >= 3.12, so it is optional: if it is skipped, use
# Runtime > Restart session after editing pinn_tailrisk/.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except ModuleNotFoundError:
    print("autoreload unavailable here; restart the session after editing the library")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pinn_tailrisk import plotting as P
from pinn_tailrisk.notebook import show_figures, load_metrics
P.use_style()

## 1. Method in one paragraph
**Physics vs data-only.** Same d-asset family and network as Stage 4 (d = 2 and 5). The PINN learns from the PIDE and
the exact density at t₀ only. The **data-only NN** is the identical network, optimiser, schedule, step count and
normaliser estimator, but its loss is the negative log-likelihood of N exact samples spread over 16 times in
[t₀, T] (13 log-spaced + the 1/5/10-day horizons), with 20% held out for early stopping. N = 10³, 10⁴, 10⁵, 10⁶,
three seeds each. **Monte Carlo on the same samples** uses the N/16 samples at each horizon directly. All models are
weighted on the *same* importance draws, so the gaps are model gaps. The output is the N at which data alone matches
the PINN.

**Failure cases** (d = 2, one change each): λ = 50/yr with the default proposal (k ≤ 3 jumps), the same with the
proposal widened to k ≤ 10, and t₀ = 0.05 day. Collected from earlier stages: d = 10 (Stage 4) and the worst θ of the
d = 5 amortised model (Stage 5). The far-tail probe shows why Stage 6 needed the r_max trust region.

In [ ]:
print(open('configs/stage7_ablation.yaml').read())

## 2. Run (resumable)
Estimated, not yet measured, on a T4: **≈ 30 min for d = 2, ≈ 35 min for d = 5** (PINN 10–13 min + 12 data-only
fits of ~1.5 min), **≈ 11 min per failure case**, summary < 1 min; about 1.7 h in total. Finished parts are skipped;
inside the data part, finished data-only fits (`results/stage7_physics_vs_data_d<d>/runs/`) and the trained PINN
(`model.pt`) are reused, so a disconnect loses at most one fit.

In [ ]:
import pathlib
for d in (2, 5):
    if pathlib.Path(f"results/stage7_physics_vs_data_d{d}/metrics.json").exists():
        print("skip physics vs data, d =", d); continue
    !python scripts/stage7_ablation.py --part data --d {d}
for case in ("high_lam", "high_lam_kfix", "tiny_t0"):
    if pathlib.Path(f"results/stage7_failure_{case}/metrics.json").exists():
        print("skip", case); continue
    !python scripts/stage7_ablation.py --part failure --case {case}
!python scripts/stage7_ablation.py --part summary

# d = 10 physics vs data: not pursued (Stage 4 breakdown point). Skeleton:
# !python scripts/stage7_ablation.py --part data --d 10 --steps 40000

## 3. Figures and what to check

| Figure | Pass criterion |
|---|---|
| physics_vs_data fig02_es_vs_samples | the blue PINN line (no samples) sits below the red data-only curve up to a large N; Monte Carlo on the same samples is the honest data baseline |
| physics_vs_data fig03_density_slices | data-only density follows the exact one where samples exist and departs in the far tail; the PINN follows it throughout |
| physics_vs_data fig04_tail_density_error | data-only tail error falls with N; the PINN's (horizontal) is reached only at large N |
| failure_high_lam fig02 | large ES error at 5–10 days (k ≤ 3 proposal misses the 4+ jump region); `high_lam_kfix` restores ≲ 1–2% |
| failure_tiny_t0 fig02 | whatever happens is reported: errors at 1 day show whether the sharper start hurts |
| final fig01_scorecard | one row per stage/setting; PINN (blue) vs its baseline; 1% reference line |
| final fig02_failure_cases | (a) where the error leaves the 1% band; (b) PINN log p − log q rises above 0 in the far tail, the exact density does not |

In [ ]:
for st in ["stage7_physics_vs_data_d2", "stage7_physics_vs_data_d5", "stage7_failure_high_lam",
           "stage7_failure_high_lam_kfix", "stage7_failure_tiny_t0", "stage7_final"]:
    print("========", st)
    show_figures(st)

## 4. Numbers

In [ ]:
for d in (2, 5):
    m = load_metrics(f"stage7_physics_vs_data_d{d}")
    print(f"== d = {d}: PINN trained {m['physics_train_time_s'] / 60:.0f} min; data-only fit ~{m['data_only_train_s_median']:.0f} s")
    print("   N at which data-only matches the PINN's ES99 error:", m["n_to_match_physics_es99"])
    a = pd.DataFrame(m["data_only_es"]); a = a[a.alpha == 0.99]
    display(a.pivot_table(index=["model", "N"], columns="h", values="median").style.format("{:.2%}"))
    p = pd.DataFrame(m["physics_es"]); p = p[p.alpha == 0.99]
    display(p.pivot_table(index="model", columns="h", values="es_abs").style.format("{:.2%}"))
print("== failure cases (ES99 error, median over portfolios)")
display(pd.read_csv("results/stage7_final/failure_cases.csv").style.format(precision=4))
print("== scorecard")
display(pd.read_csv("results/stage7_final/scorecard.csv").style.format(precision=4))

## 5. Package the outputs
Stage 7 figures and results (without model weights) in one zip: on Kaggle it appears under **Output**; on Colab it is written to the repo folder.

In [ ]:
import pathlib, zipfile
zpath = pathlib.Path("/kaggle/working" if ON_KAGGLE else ".") / "stage7_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for top in ("figures", "results"):
        for f in sorted(pathlib.Path(top).glob("stage7_*/**/*")):
            if f.is_file() and f.suffix != ".pt" and "_quick" not in str(f):
                z.write(f)
print(zpath.resolve(), f"{zpath.stat().st_size / 1e6:.1f} MB,", len(zipfile.ZipFile(zpath).namelist()), "files")

## 6. Release checklist
- [ ] all notebooks 00–08 run top to bottom from a fresh Colab
- [ ] `figures/` and `results/` committed (zip from Colab → local → push)
- [ ] README reproduce table and PROJECT_HANDOFF status current
- [ ] tag the thesis version: `git tag -a v1.0 -m "thesis results" && git push --tags`